In [2]:
import folium, json
from IPython.display import display
import requests

In [2]:
coords_observedArea = [
    [53.5805445, 9.7772179],
    [53.5805446, 9.777218],
    [53.5811757, 9.7800438],
    [53.5805445, 9.7772179],
]
coords_Locations = [
    [[53.5805445, 9.7772179], [53.5805535, 9.7772171]],
    [[53.5805446, 9.7772173], [53.5804716, 9.7772229]],
    [[53.5804715, 9.7772235], [53.5804625, 9.7772243]],
]


# Create a map object centered at a specific latitude and longitude
map_object = folium.Map(location=coords_observedArea[0], zoom_start=12)

colors = [
    "red",
    "blue",
    "green",
    "purple",
    "orange",
    "darkred",
    "lightred",
    "beige",
    "darkblue",
    "darkgreen",
    "cadetblue",
    "darkpurple",
    "white",
    "pink",
    "lightblue",
    "lightgreen",
    "gray",
    "black",
]
color_counter = 0
# Add markers to the map
for coord in coords_observedArea:
    folium.CircleMarker(
        location=coord,
        radius=5,
        color=colors[color_counter],
        fill=True,
        fill_color=colors[color_counter],
        opacity=0.5,
    ).add_to(map_object)

color_counter += 1


def display_multilinestring(coordinates: list):
    for LineString in coordinates:
        for coord in LineString:
            folium.CircleMarker(
                location=coord,
                radius=5,
                color=colors[color_counter],
                fill=True,
                fill_color=colors[color_counter],
                opacity=0.5,
            ).add_to(map_object)
        color_counter += 1


# Display the map
# display(map_object)

In [3]:
base_url = "https://tld.iot.hamburg.de/v1.1/Datastreams"
params = {
    "$filter": "properties/layerName eq 'primary_signal' and Thing/properties/trafficLightsID eq '1556'"
}

response = requests.get(base_url, params=params)
thing_url = ""
if response.status_code == 200:
    data = response.json()

    # get datastream of a single traffic light
    tl_stream_url = data["value"][0]["@iot.selfLink"]
    tl_data = requests.get(tl_stream_url).json()

    # print(json.dumps(tl_data, indent=2))
    # Datastream does not show related ingress egress lane

    thing_url = data["value"][0]["Thing@iot.navigationLink"]

In [16]:
locations_data = requests.get(thing_url + "/Locations").json()
# print(json.dumps(locations_data, indent=2))
coordinates = locations_data["value"][0]["location"]["geometry"]["coordinates"]
print(coordinates)

map_object = folium.Map(location=[53.5605621, 9.9903945], zoom_start=25)
for idx, LineString in enumerate(coordinates[2:]):
    for coord in LineString:
        folium.CircleMarker(
            location=(coord[1], coord[0]),
            radius=5,
            color=colors[idx],
            fill=True,
            fill_color=colors[idx],
            opacity=0.5,
        ).add_to(map_object)
display(map_object)

[[[9.9903945, 53.5605621], [9.9903525, 53.5605993], [9.9902972, 53.5606509]], [[9.9903547, 53.5605982], [9.9904327, 53.5605292], [9.9904327, 53.5605292], [9.9903485, 53.5603945]], [[9.9903805, 53.5604501], [9.9903463, 53.5603953], [9.9902725, 53.5602814]]]


In [17]:
thing_data = requests.get(thing_url + "/Datastreams").json()
print(json.dumps(thing_data, indent=2))

{
  "value": [
    {
      "@iot.selfLink": "https://tld.iot.hamburg.de/v1.1/Datastreams(70477)",
      "@iot.id": 70477,
      "name": "Primary signal heads at 1556_4",
      "description": "Datastream to broadcast the lane connection's signal value of a signal group",
      "observationType": "http://www.opengis.net/def/ogc-om/OM_CategoryObservation",
      "unitOfMeasurement": {
        "name": "Signal Status ",
        "symbol": "[0-6,9]",
        "definition": "0=dark,1=red,2=amber,3=green,4=red-amber,5=amber-flashing,6=green-flashing,9=unknown"
      },
      "observedArea": {
        "type": "LineString",
        "coordinates": [
          [
            9.9903943,
            53.5605618
          ],
          [
            9.9903945,
            53.5605621
          ]
        ]
      },
      "phenomenonTime": "2009-10-28T19:32:44Z/2024-10-23T12:05:07Z",
      "properties": {
        "metadata": "https://registry.gdi-de.org/id/de.hh/fa5588ec-bc36-4070-8871-6bf69365fe79",
       